# Citi Bike: automated model comparison

**Contribution owner: Tomislav Novosel.** Implementation, experiments and initial explanations were prepared with AI assistance on 9 October 2026. Human review and substantive changes should be recorded when they happen; see [the assistance record](../docs/AI_USE.md).

These are development experiments, not final-test results. The training/tuning dates end in June; July-September is development validation. October-December has not been scored.

The searches were executed with the helper scripts and their actual reports are saved. By default this notebook reviews those reports quickly. Set `CITIBIKE_RETRAIN=1` or use the runner's `--train` option to execute the search again. The XGBoost search needs a CUDA GPU; viewing results and running the saved app do not.


FLAML tries configurations automatically across LightGBM, random forest, Extra Trees and XGBoost. It uses the same chronological training folds as the manual searches.

In [5]:
from pathlib import Path
import os, sys, json
ROOT = Path.cwd()
while not (ROOT / 'src/citibike_forecast.py').exists():
    if ROOT == ROOT.parent:
        raise RuntimeError('Open the notebook inside the repository.')
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / 'src'))
import pandas as pd
import numpy as np
from citibike_forecast import REPORTS, MODELS, development_data, chronological_cv
RUN_TRAINING = os.environ.get('CITIBIKE_RETRAIN') == '1'
def report(name):
    return json.loads((REPORTS / (name + '_run.json')).read_text(encoding='utf8'))
print('Mode:', 'execute training' if RUN_TRAINING else 'review actual saved experiments')
from citibike_forecast import run_automl
if RUN_TRAINING:
    run_automl(180)
automl = report('automl')
display(pd.Series(automl['search']))
trials = pd.read_json(REPORTS / 'automl_search.jsonl', lines=True)
trials = trials.dropna(subset=['learner'])
display(trials.groupby('learner').agg(trials=('validation_loss','size'), best_training_cv_mae=('validation_loss','min')))


Mode: review actual saved experiments


tool                                                               FLAML AutoML
estimator_families                              [lgbm, rf, extra_tree, xgboost]
best_family                                                          extra_tree
best_parameters               {'n_estimators': 7, 'max_features': 0.96064188...
best_cv_mae                                                        13506.274234
elapsed_seconds                                                        5.741776
budget_seconds                                                              180
max_trials                                                                   80
validation_used_for_search                                                False
scope                         Automatic configuration search on training dat...
dtype: object

,trials,best_training_cv_mae
learner,,
extra_tree,19,13506.274234
lgbm,23,14893.919693
rf,10,13774.150859
xgboost,28,14189.478190


The best search score is not its independent performance. Here we compare the selected configuration on the later July-September validation period.

In [6]:
display(pd.Series(automl['validation']))
print('FLAML considered all four families in 80 actual trials. It did not beat the simpler alternatives on this validation period.')
print('Keep this unsuccessful experiment: it explains why automatic search is not automatically the best result.')


FLAML considered all four families in 80 actual trials. It did not beat the simpler alternatives on this validation period.
Keep this unsuccessful experiment: it explains why automatic search is not automatically the best result.


mae     22331.212727
rmse    26534.980362
wape        0.185167
bias   -11451.169448
days       92.000000
dtype: float64